<a href="https://colab.research.google.com/github/Luizadraeger/URBAN-DATA-ANALYTICS/blob/main/pipeline_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Célula 3: Importa livrarias
import ee
import geemap
import geopandas as gpd
import os
import time
import pandas as pd

In [2]:
# Célula 4: Autenticação e inicialização dos serviços Google
# Esta célula configura o acesso ao Google Drive e ao Google Earth Engine (EE).

# PASSOS NECESSÁRIOS:
# 1. Monte o Google Drive para leitura/escrita de arquivos.
# 2. Autentique sua conta Google para uso do Earth Engine.
# 3. Inicialize o Earth Engine com um projeto válido do Google Cloud.

# IMPORTANTE:
# - Antes de executar, crie um projeto no Google Cloud Console.
# - Ative a API do Earth Engine nesse projeto.
# - Substitua 'thermal-luizadraeger' pelo ID do seu projeto.
# - Durante a execução, será solicitado um token de autenticação no Colab.

from google.colab import drive

# Monta o Google Drive no ambiente do Colab
# force_remount=True garante remontagem caso já esteja conectado
try:
    drive.mount('/content/drive', force_remount=True)
except:
    drive.mount('/content/drive')

# ─── Autenticação no Google Earth Engine ─────────────────────────────────────
# Abre um fluxo de login para autorizar o acesso ao EE
import ee
ee.Authenticate()

# Inicializa o Earth Engine com o projeto especificado
# Substitua pelo seu próprio ID de projeto no Google Cloud
ee.Initialize(project='thermal-luizadraeger')

Mounted at /content/drive


In [3]:
def buildfooprint(lat, lon, radius_m):
    """
    Extrai footprints e alturas de edifícios usando Google Open Buildings.
    """
    try:
        point = ee.Geometry.Point([lon, lat])
        region = point.buffer(radius_m).bounds()

        # Dataset de Alturas (Temporal V1)
        temporal_col = ee.ImageCollection('GOOGLE/Research/open-buildings-temporal/v1') \
                         .filterBounds(region) \
                         .sort('system:time_start', False)
        latest_raster = temporal_col.first()

        # Polígonos de Footprints (V3)
        buildings_fc = ee.FeatureCollection('GOOGLE/Research/open-buildings/v3/polygons') \
                         .filterBounds(region)

        # Amostragem de alturas por polígono
        sampled_fc = latest_raster.select('building_height').reduceRegions(
            collection=buildings_fc,
            reducer=ee.Reducer.median(),
            scale=1.0
        )

        features = sampled_fc.getInfo().get('features', [])
        building_list = []

        for f in features:
            props = f.get('properties', {})
            geom = f.get('geometry')
            if not geom: continue

            h_val = props.get('median')
            # Define altura padrão de 3.5m caso não encontre o dado
            height = float(h_val) if (h_val and h_val > 0) else 3.5

            # Extração de coordenadas para o Grasshopper
            coords = geom['coordinates'][0] if geom['type'] == 'Polygon' else geom['coordinates'][0][0]

            building_list.append({
                "geometry": coords,
                "height": height
            })

        return building_list
    except Exception as e:
        raise Exception(f"Erro no processamento GEE: {str(e)}")

In [4]:
%cd /content
!rm -rf URBAN-DATA-ANALYTICS

/content


In [5]:
!git clone https://github.com/Luizadraeger/URBAN-DATA-ANALYTICS.git

Cloning into 'URBAN-DATA-ANALYTICS'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 84 (delta 32), reused 6 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 160.24 KiB | 4.45 MiB/s, done.
Resolving deltas: 100% (32/32), done.


In [8]:
def get_river_data(lat, lon, radius_m):
    """
    Recorta o rio e gera zonas de proteção (APP) de 10m e 50m.
    """
    import os
    import geopandas as gpd
    from shapely.geometry import Point, LineString

    base_path = '/content/URBAN-DATA-ANALYTICS/references'

    # 1. Localização dinâmica da pasta
    if not os.path.exists(base_path):
        return []

    target_folder = None
    for f in os.listdir(base_path):
        if "Hidrografia" in f or "Paquequer" in f:
            target_folder = os.path.join(base_path, f)
            break

    if not target_folder:
        return []

    try:
        # 2. Carregamento do Shapefile
        shp_file = next((f for f in os.listdir(target_folder) if f.endswith('.shp')), None)
        if not shp_file:
            return []

        gdf = gpd.read_file(os.path.join(target_folder, shp_file))
        if gdf.crs != 'EPSG:4326':
            gdf = gdf.to_crs('EPSG:4326')

        # 3. Recorte Espacial
        center_pt = Point(lon, lat)
        buffer_deg = radius_m / 111320.0
        aoi_geom = center_pt.buffer(buffer_deg)
        river_clip = gdf[gdf.intersects(aoi_geom)].copy()

        output_data = []

        if not river_clip.empty:
            # Fatores de conversão para metros -> graus (aproximado)
            deg_factor = 111320.0

            for _, row in river_clip.iterrows():
                geom = row.geometry
                lines = []
                if geom.geom_type == 'LineString':
                    lines.append(geom)
                elif geom.geom_type == 'MultiLineString':
                    lines.extend(list(geom.geoms))

                for line in lines:
                    # A. Eixo do Rio (Original)
                    output_data.append({
                        "type": "LineString",
                        "coordinates": list(line.coords)
                    })

                    # B. Zona de 10 metros
                    risk_10 = line.buffer(10 / deg_factor)
                    if not risk_10.is_empty:
                        output_data.append({
                            "type": "RiskZone_10",
                            "coordinates": list(risk_10.exterior.coords)
                        })

                    # C. Zona de 50 metros
                    risk_50 = line.buffer(50 / deg_factor)
                    if not risk_50.is_empty:
                        output_data.append({
                            "type": "RiskZone_50",
                            "coordinates": list(risk_50.exterior.coords)
                        })

            print(f"✅ Rio processado com faixas de 10m e 50m.")
            return output_data
        return []

    except Exception as e:
        print(f"Erro: {e}")
        return []

In [9]:
def get_terrain_data(lat, lon, radius_m):
    """
    Coleta pontos de altitude dentro de um raio especificado.
    """
    try:
        # Define o ponto e a área de busca
        point = ee.Geometry.Point([lon, lat])
        region = point.buffer(radius_m).bounds()

        # Carrega o dataset NASADEM (resolução de ~30m)
        dem = ee.Image('NASA/NASADEM_HGT/001').select('elevation')

        # Amostragem de pontos dentro da ROI (escala de 30 metros para Teresópolis)
        sample = dem.sampleRegions(
            collection=ee.FeatureCollection([region]),
            scale=30,
            geometries=True
        )

        features = sample.getInfo().get('features', [])
        terrain_pts = []

        for f in features:
            terrain_pts.append({
                "type": "TerrainPoint",
                "coordinates": f['geometry']['coordinates'], # [lon, lat]
                "elevation": float(f['properties']['elevation'])
            })

        print(f"✓ Relevo: {len(terrain_pts)} pontos de altitude coletados.")
        return terrain_pts
    except Exception as e:
        print(f"⚠️ Erro no Pipeline de Relevo: {e}")
        return []

In [10]:
# Na sua rota unificada, após coletar 'buildings' e 'river_data':
def check_risk_overlap(buildings, river_data):
    from shapely.geometry import Polygon, LineString

    # 1. Cria a geometria da zona de risco (APP de 30m)
    river_lines = []
    for r in river_data:
        if r.get('type') == 'LineString':
            river_lines.append(LineString(r['coordinates']))

    if not river_lines: return buildings

    # Une todas as linhas do rio e cria o buffer de 30m
    full_river = LineString([pt for line in river_lines for pt in line.coords]) # Simplificado
    risk_area = full_river.buffer(30 / 111320.0)

    # 2. Verifica cada prédio
    for b in buildings:
        poly = Polygon(b['geometry'])
        if poly.intersects(risk_area):
            b['is_at_risk'] = True # Marca o prédio
        else:
            b['is_at_risk'] = False

    return buildings